In [1]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.metrics import confusion_matrix, f1_score, make_scorer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay, roc_auc_score, average_precision_score
)

## Read Data CSV

In [2]:
sessions = pd.read_csv("Raw_Data/challenge2_train.csv")

display(sessions.head())
sessions.info()

print()
display(sessions["Revenue"].value_counts(dropna=False))
print()
display(sessions["Revenue"].value_counts(normalize=True, dropna=False))

,ID,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,1,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,3,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
2,4,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
3,7,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.4,Feb,2,4,3,3,Returning_Visitor,False,False
4,9,0,0.0,0,0.0,2,37.000000,0.00,0.10,0.0,0.8,Feb,2,2,2,3,Returning_Visitor,False,False


<class 'pandas.DataFrame'>
RangeIndex: 9247 entries, 0 to 9246
Data columns (total 19 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       9247 non-null   int64  
 1   Administrative           9247 non-null   int64  
 2   Administrative_Duration  9247 non-null   float64
 3   Informational            9247 non-null   int64  
 4   Informational_Duration   9247 non-null   float64
 5   ProductRelated           9247 non-null   int64  
 6   ProductRelated_Duration  9247 non-null   float64
 7   BounceRates              9247 non-null   float64
 8   ExitRates                9247 non-null   float64
 9   PageValues               9247 non-null   float64
 10  SpecialDay               9247 non-null   float64
 11  Month                    9247 non-null   str    
 12  OperatingSystems         9247 non-null   int64  
 13  Browser                  9247 non-null   int64  
 14  Region                   9247 non-n

Revenue
False    7816
True     1431
Name: count, dtype: int64

Revenue
False    0.845247
True     0.154753
Name: proportion, dtype: float64

## Train/Validation Splits 

In [3]:
X = sessions.drop(columns=["Revenue", "ID"])
y = sessions["Revenue"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

display(pd.DataFrame({
    "development": y_train.value_counts(normalize=True),
    "holdout": y_valid.value_counts(normalize=True),
}))

,development,holdout
Revenue,,
False,0.845208,0.845405
True,0.154792,0.154595


## Model Pipeline

In [14]:
categorical = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend",
]

numerical = [column for column in X_train.columns if column not in categorical]

# 2. Build leakage-free preprocessing per column type
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    # Yeo-Johnson maps skewed continuous features to Gaussian bell curves
    ('power', PowerTransformer(method='yeo-johnson', standardize=True))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    # Group rare categories (<1% frequency) to prevent zero-variance overfitting
    ('ohe', OneHotEncoder(
        handle_unknown='infrequent_if_exist',
        min_frequency=0.01,
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical),
        ('cat', cat_pipeline, categorical)
    ],
    remainder='drop'
)

### Logistic Regression

In [15]:
logistic_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=1000)),
])

fitted_logistic = clone(logistic_pipeline).fit(X_train, y_train)

feature_names = fitted_logistic.named_steps["prep"].get_feature_names_out()
weights = fitted_logistic.named_steps["model"].coef_[0]

logistic_weights = pd.DataFrame({
    "Feature": feature_names,
    "Weight": weights,
})

display(logistic_weights.head(10).round(3))

,Feature,Weight
0,num__Administrative,-0.371
1,num__Administrative_Duration,0.066
2,num__Informational,0.196
3,num__Informational_Duration,-0.240
4,num__ProductRelated,-0.121
5,num__ProductRelated_Duration,0.092
6,num__BounceRates,-0.195
7,num__ExitRates,-0.363
8,num__PageValues,1.625
9,num__SpecialDay,-0.035


### Naïve Bayes

In [6]:
Bayes_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", BernoulliNB())
])

Bayes_pipe.fit(X_train, y_train)

feature_names = Bayes_pipe.named_steps['prep'].get_feature_names_out()

feature_prob = np.exp(Bayes_pipe.named_steps['clf'].feature_log_prob_)

learned_probabilities = pd.DataFrame({
    'Feature': feature_names,
    'P(Feature=1 | y=0)': feature_prob[0],
    'P(Feature=1 | y=1)': feature_prob[1]
})

learned_probabilities.head(10).round(2)


,Feature,P(Feature=1 | y=0),P(Feature=1 | y=1)
0,num__Administrative,0.50,0.74
1,num__Administrative_Duration,0.47,0.71
2,num__Informational,0.19,0.33
3,num__Informational_Duration,0.17,0.30
4,num__ProductRelated,0.48,0.73
5,num__ProductRelated_Duration,0.47,0.74
6,num__BounceRates,0.41,0.18
7,num__ExitRates,0.49,0.15
8,num__PageValues,0.11,0.82
9,num__SpecialDay,0.12,0.04


In [7]:
# Predictions for training and test sets
y_train_pred = Bayes_pipe.predict(X_train)
y_test_pred = Bayes_pipe.predict(X_valid)
y_test_prob = Bayes_pipe.predict_proba(X_valid)[:, 1]

# Compare the main classification metrics
metrics_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Training": [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred)
    ],
    "Validation": [
        accuracy_score(y_valid, y_test_pred),
        precision_score(y_valid, y_test_pred),
        recall_score(y_valid, y_test_pred),
        f1_score(y_valid, y_test_pred)
    ]
})

metrics_comparison.round(3)

,Metric,Training,Validation
0,Accuracy,0.836,0.808
1,Precision,0.478,0.421
2,Recall,0.679,0.654
3,F1 Score,0.561,0.512


### K-Nearest Neighbors

In [8]:
knn_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=5)),
])

fitted_knn = clone(knn_pipeline).fit(X_train, y_train)
knn_model = fitted_knn.named_steps["model"]
print(knn_model.weights)

session = X_train.iloc[[0]]
transformed_session = fitted_knn.named_steps["prep"].transform(session)
distances, indices = knn_model.kneighbors(transformed_session)

display(pd.DataFrame({
    "Distance": distances[0],
    "Purchased": y_train.iloc[indices[0]].to_numpy(),
}))

uniform


,Distance,Purchased
0,0.000000,False
1,1.568383,False
2,1.770997,False
3,1.857021,False
4,1.955126,False


## Support Vector Machine

In [9]:

svmlinear_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(kernel="linear", C=1.0, class_weight="balanced"))
])

svmrbf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    ))
])

fitted_svmlinear = clone(svmlinear_pipe).fit(X_train, y_train)
fitted_svmrbf = clone(svmrbf_pipe).fit(X_train, y_train)

svmlinear_model = fitted_svmlinear.named_steps["model"]
svmrbf_model = fitted_svmrbf.named_steps["model"]

print("Linear SVM support vectors:", svmlinear_model.n_support_)
print("RBF SVM support vectors:", svmrbf_model.n_support_)


Linear SVM support vectors: [1878  345]
RBF SVM support vectors: [1953  548]


## Model Comparison

In [11]:
model_results = []

pipelines = {
    "Logistic Regression": logistic_pipeline,
    "Naïve Bayes": Bayes_pipe,
    "K-Nearest Neighbors": knn_pipeline,
    "Linear SVM": svmlinear_pipe,
    "RBF SVM": svmrbf_pipe
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_purchase = make_scorer(f1_score, pos_label=True, zero_division=0)

for model, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=f1_purchase,
        return_train_score=True,
    )

    model_results.append({
        "Model": model,
        "Mean training F1": scores["train_score"].mean(),
        "Mean CV F1": scores["test_score"].mean(),
    })
    
display(pd.DataFrame(model_results).round(3))

,Model,Mean training F1,Mean CV F1
0,Logistic Regression,0.674,0.662
1,Naïve Bayes,0.563,0.558
2,K-Nearest Neighbors,0.729,0.633
3,Linear SVM,0.681,0.680
4,RBF SVM,0.712,0.672


## Test Predictions

In [13]:
winning_pipeline = knn_pipeline
final_model = clone(winning_pipeline).fit(X, y)

test_sessions = pd.read_csv("Raw_Data/challenge2_test.csv")
X_test = test_sessions[X.columns]

predictions = pd.DataFrame({
    "ID": test_sessions["ID"],
    "Revenue": final_model.predict(X_test),
})

assert len(predictions) == 3083
assert predictions["ID"].is_unique
assert predictions["Revenue"].isin([True, False]).all()

predictions.to_csv("predictions.csv", index=False)
display(predictions.head())

,ID,Revenue
0,2,False
1,5,False
2,6,False
3,8,False
4,10,False
